# 树定价、闭式与隐波反演

先倒推每个节点，再观察离散误差和反演敏感性。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
rng = np.random.default_rng(20260907)
np.set_printoptions(precision=5, suppress=True)

统一无股息欧式看涨参数，写出闭式与 CRR 的全部倒推。

In [ ]:
S=100.;K=100.;r=.02;sigma=.2;T=1.
def call_price(vol):
    d1=(np.log(S/K)+(r+.5*vol**2)*T)/(vol*np.sqrt(T));d2=d1-vol*np.sqrt(T)
    return S*stats.norm.cdf(d1)-K*np.exp(-r*T)*stats.norm.cdf(d2)
reference=call_price(sigma)
for n in [1,16,64,256]:
    dt=T/n;u=np.exp(sigma*np.sqrt(dt));d=1/u;p=(np.exp(r*dt)-d)/(u-d)
    terminal=S*u**np.arange(n+1)*d**(n-np.arange(n+1));values=np.maximum(terminal-K,0)
    for step in range(n):values=np.exp(-r*dt)*((1-p)*values[:-1]+p*values[1:])
    print(n,'probability:',p,'price:',values[0],'error:',values[0]-reference)
print('closed form:',reference)

单调性给二分反演，改动报价观察隐波变化。

In [ ]:
quote=reference;low=.001;high=2.
for step in range(50):
    mid=(low+high)/2
    if call_price(mid)<quote:low=mid
    else:high=mid
print('implied volatility:',(low+high)/2,'price residual:',call_price((low+high)/2)-quote)

## 自己试一试

树的步数翻倍，误差是否必然严格减半？

## 反馈

不必，CRR 可能奇偶振荡，收敛阶不代表每一对网格都精确按同一比例下降。

参数改变后应重新解释结果，不要求复现某次随机实验的小数。